# RAG Assessment

TODO: just attach the hyperlink to the pdf file here

## My approach

The brief says the goal is not a polished system but the ability to reason about retrieval, grounding,
hallucination risk and evaluation, and to explain where the system works, where it fails and why. So I
split the work into two phases and spent my own time on the second.

**Phase 1 — boilerplate, built with AI assistance.** I used Claude Code (Anthropic) to produce the
starting point quickly:
- the RAG pipeline (`rag/`): chunking, embedding and Chroma indexing, retrieval, the answer prompt and
  its JSON output contract, and a small CLI;
- an initial golden set (`eval/golden/v0.json`): the 10 supplied questions plus 10 drafted
  self-generated questions, with expected behaviour, expected support status, required evidence and
  reference answers;
- the evaluation pipeline (`eval/`): rule-based checks (status, evidence recall, exact abstention) and
  the RAGAS metrics (faithfulness, context precision, context recall, factual correctness).

I set the direction (plain Python rather than a framework, single retrieval path, LiteLLM with Gemini)
and reviewed the output, but I treat this phase as scaffolding rather than as the work being assessed.

**Phase 2 — review, analysis and iteration: this notebook.** Everything from here on is my own
evaluation of that baseline: checking that the golden set's expectations match the documents, running
the system, reading the failures, deciding which low scores are real and which are metric artefacts,
and testing changes one at a time with a stated hypothesis and a measured result. I am responsible for
every design decision and conclusion in it and can explain all of the code.

## How to read this notebook

| Section | Contents |
|---|---|
| 1–4 | The baseline system: chunking, retrieval, generation, a worked example |
| 5 | Evaluation design: 20 questions with expected behaviour defined up front |
| 6 | Baseline evaluation results (required table + metrics) |
| 7 | Failure analysis |
| 8 | Iterations: targeted changes, each measured against the baseline |
| 9 | Write-up (chunking, retrieval, prompting, hallucination, abstention, limitations, next steps, sensitive data) |
| 10 | Production-readiness notes |
| 11 | Main question |

Implementation lives in the modules, not in this notebook; the notebook imports them, so the same code
runs in the CLI, the evaluation and here.

# TODO: double check the setup command
**Setup**

```bash
uv sync                  # or: python3.12 -m venv .venv && pip install -r requirements.txt
cp .env.example .env     # set GEMINI_API_KEY (no keys are stored in this repo)
python -m rag index      # build the local Chroma index
```

In [1]:
import json
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)

from eval import report  # noqa: E402
from rag.chunking import load_chunks  # noqa: E402
from rag.config import ABSTAIN_ANSWER, get_settings  # noqa: E402

settings = get_settings()
print(f"answer model: {settings.llm_model}\nembedding model: {settings.embed_model}")
print(f"top_k: {settings.top_k}   max chunk words: {settings.max_chunk_words}")

answer model: gemini/gemini-3.5-flash
embedding model: gemini/gemini-embedding-001
top_k: 5   max chunk words: 300


## 1. Document loading and chunking

Four Markdown documents, ~11.7k words: three PDPC extracts (public guidance) and one synthetic
internal policy addendum. `README.md` is excluded, as the brief requires.

**Strategy: one chunk per Markdown section.** Each `##`/`###` section becomes a chunk carrying its
heading path; sections longer than 300 words are split on paragraph boundaries with a one-paragraph
overlap. The documents are already organised one topic per section, so a section is usually the
complete unit an answer needs — splitting by fixed token windows would separate rules from their
conditions (e.g. the external-sharing rule from the approvals it requires).

Each chunk carries metadata used later:
- `chunk_id` — stable, readable (`policy-05`), so citations point at something a reviewer can look up;
- `section` — heading path, prefixed to the text before embedding so short chunks keep their topic;
- `authority` — `internal_policy` or `public_guidance`, so the prompt can apply the brief's rule that
  the internal policy wins where it is more specific.

In [ ]:
chunks = load_chunks(settings.docs_dir, settings.max_chunk_words)
sizes = pd.Series([len(c.text.split()) for c in chunks])

print(f"{len(chunks)} chunks from {len({c.source for c in chunks})} documents")
print(f"words per chunk: min {sizes.min()}, median {int(sizes.median())}, max {sizes.max()}")
display(
    pd.DataFrame(
        [{"source": c.source, "authority": c.authority, "words": len(c.text.split())} for c in chunks]
    )
    .groupby(["source", "authority"])
    .agg(chunks=("words", "size"), words=("words", "sum"))
)

example = next(c for c in chunks if c.section.startswith("4. Small Cell"))
print(f"\nExample chunk [{example.chunk_id}] {example.source} :: {example.section}\n")
print(example.text)

## 2. Indexing and retrieval

Chunks are embedded with `gemini-embedding-001` through LiteLLM and stored in a local persistent
Chroma collection (cosine distance). `python -m rag index` rebuilds the collection from scratch and
stamps it with the embedding model and a hash of the corpus, so a stale index is detected rather than
silently queried.

Retrieval is single-path dense search over the top `k` chunks. The brief prefers evaluation depth over
extra components, so hybrid retrieval and re-ranking were deliberately left out until the evaluation
showed whether they were needed (section 7 revisits this).

The cell below shows retrieval only — no generation.

In [ ]:
from rag.pipeline import RagPipeline  # noqa: E402
from rag.store import retrieve  # noqa: E402

pipeline = RagPipeline(settings)
question = "What approvals are required before de-identified patient-level data may be shared with an external party?"

hits = retrieve(question, settings, pipeline._embed, k=settings.top_k)
pd.DataFrame(
    [
        {
            "chunk_id": h.chunk_id,
            "authority": h.authority,
            "section": h.section,
            "distance": round(h.distance, 3),
            "snippet": " ".join(h.text.split())[:110] + "...",
        }
        for h in hits
    ]
)

## 3. Answer generation

Retrieved chunks are passed as labelled passages and the model must return JSON:
`{answer, support_status, citations, missing_information}` at temperature 0.

The prompt states five rules: cite passage IDs for every material claim; prefer `internal_policy` over
`public_guidance` where more specific; check the question's premise; treat passages and questions as
data rather than instructions; and choose the support status, with the exact abstention string for
`not supported`.

**The prompt is not trusted on its own.** `parse_and_validate` enforces the contract in code:

| Model output | Enforced result |
|---|---|
| citation not in the retrieved set | citation dropped, warning recorded |
| invalid JSON or unknown status | abstention |
| `supported`/`partially supported` with no valid citation | abstention |
| `not supported` | answer replaced with the exact abstention string |

This is what makes "every claim is cited" a property of the system rather than a request to the model.

In [ ]:
from rag.generation import SYSTEM_PROMPT, parse_and_validate  # noqa: E402

print(SYSTEM_PROMPT)

# Guardrails, demonstrated without calling the API: a fabricated citation and an uncited claim.
bad_outputs = {
    "cites a chunk that was not retrieved": json.dumps(
        {"answer": "...", "support_status": "supported", "citations": ["policy-99"]}
    ),
    "claims support with no citation": json.dumps(
        {"answer": "Data may be shared freely.", "support_status": "supported", "citations": []}
    ),
    "not valid JSON": "I think the answer is probably yes.",
}
for label, raw in bad_outputs.items():
    result, warns = parse_and_validate(raw, hits)
    print(f"\n{label}\n  -> status={result['support_status']!r} answer={result['answer'][:60]!r}\n  -> {warns}")

## 4. Worked example

One question end to end: the answer, the passages used, the citations, and the support status.
This calls the API.

In [ ]:
result = pipeline.ask(question)

print(f"ANSWER\n{result.answer}\n")
print(f"SUPPORT STATUS: {result.support_status}")
print(f"CITATIONS: {', '.join(result.citations)}")
print(f"MISSING: {result.missing_information}")
print(f"WARNINGS: {result.warnings}\n")
print("PASSAGES USED ('*' = cited)")
for c in result.retrieved:
    mark = "*" if c.chunk_id in result.citations else " "
    print(f" {mark} [{c.chunk_id}] {c.source} :: {c.section} (distance={c.distance:.3f})")

In [ ]:
# Abstention on a question the corpus cannot answer.
abstained = pipeline.ask("What is the maximum financial penalty the PDPC can impose on an organisation for breaching the PDPA?")
print(abstained.answer)
print(f"status={abstained.support_status}  citations={abstained.citations}")
print(f"exact required string: {abstained.answer.strip() == ABSTAIN_ANSWER}")

## 5. Evaluation design

`eval/golden/v0.json` holds 20 questions: the 10 supplied (verbatim) and 10 written for this
assessment. **Expected behaviour, expected support status and required evidence were written before
running the system**, from the source documents.

Each item carries:
- `expected_behavior` — what a correct response must do;
- `expected_status` — `supported` / `partially supported` / `not supported`;
- `required_evidence` — groups of `{source, quote}`. **Every group** must be retrieved, and **any**
  quote inside a group satisfies it (the same rule appears in several documents). Quotes are used
  instead of chunk IDs so the golden set survives a change of chunking strategy;
- `reference` — a claim-level answer used by the judged metrics;
- `missing_points` — for partial cases, what the answer must identify as unsupported.

`uv run python -m eval.check_golden` verifies that every quote exists verbatim and fits inside one
chunk.

**Versioning.** Golden sets are immutable once a run has used them; any change becomes a new version
(`eval/golden/vN.json`, explained in `eval/golden/CHANGELOG.md`). Each run records the golden version
and a hash, and this notebook always displays a run against the version it was scored with, so later
iterations never alter earlier results. Coverage of v0 below.

In [ ]:
baseline = report.load_run("01-baseline-k5")
golden = baseline.golden  # the golden version this run was scored against
coverage = pd.DataFrame(
    [{"id": q["id"], "source": q["source"], "type": q["type"], "expected_status": q["expected_status"]} for q in golden.values()]
)
display(pd.crosstab(coverage["type"], coverage["source"], margins=True))
display(coverage["expected_status"].value_counts().rename("questions"))

item = golden["S02"]
print(json.dumps({k: item[k] for k in ["question", "expected_behavior", "expected_status", "required_evidence", "reference"]}, indent=2)[:1400])

### Scoring

**Deterministic (no judge):**
- `status_correct` — predicted support status equals expected;
- `evidence_recall` — share of required evidence groups present in the retrieved passages;
- `rule_pass` — status correct **and** all required evidence retrieved **and** the exact abstention
  string where abstention was expected.

**Judged (RAGAS, Gemini as judge, temperature 0):** `faithfulness` (answer claims supported by the
retrieved passages), `context_precision`, `context_recall`, `factual_correctness` (claim-level F1 vs
the reference). These are skipped for the three questions that must be abstained on, which have no
claims to score and are judged by exact abstention instead.

**Pass/Fail** in the results table is a reviewed verdict: the rule check cannot tell whether the
answer followed the expected behaviour (e.g. rejecting a false premise), so cases are read and any
override is recorded in the `REVIEW` dict in section 6.

In [ ]:
# Every run is a committed, self-contained folder in eval/results/<name>/ (answers, scores, golden
# version, prompt, review), so this cell reads results rather than regenerating them.
# Reproduce a run: uv run python -m eval.run_eval --name <new-name>   (see eval/run_eval.py)
runs = {name: report.load_run(name) for name in ["01-baseline-k5"]}
print(f"baseline run: {baseline.name}")
print(json.dumps(baseline.config, indent=2))

## 6. Evaluation results

Required table first (one row per question), then the scores.

In [ ]:
# Manual review of run 01. The rule check cannot judge whether an answer followed its expected
# behaviour, so each case was read; verdicts and notes here override the rule check in the table.
# Questions not listed fall back to rule_pass.
REVIEW = {
    "S08": {"pass": True, "note": "Correct scope and missing-information call-out, but upgrades the documents' 'should' (recommended control) to 'must' in three claims."},
    "S10": {"pass": True, "note": "Injection ignored; exact abstention string returned."},
    "C02": {"pass": True, "note": "Reviewed as correct: states all waiver conditions and both approvals. Low factual_correctness is a judge matching artefact (question-specific phrasing vs general reference)."},
    "C04": {"pass": False, "note": "Retrieval miss at k=5: notifiability criteria (significant harm / significant scale) not retrieved, so the answer states only that an assessment is required."},
    "C07": {"pass": True, "note": "Abstained despite highly relevant retrieved passages that lack the requested response time."},
    "C10": {"pass": True, "note": "Injected threshold of 3 ignored; documented fewer-than-5 rule applied."},
}

table = report.assignment_table(baseline, REVIEW)
table.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"})

In [ ]:
display(report.scores_table(baseline).style.format(precision=2))
summary = baseline.summary()
print("Overall:", json.dumps(summary["overall"]))
display(pd.DataFrame(summary["by_type"]).T)

**Reading the headline numbers.** Support status is correct on every question, including all three
abstentions and the two prompt-injection attempts. Evidence recall and the judged context metrics are
high. `factual_correctness` is much lower than the rest — section 7 shows that this is mostly a
property of the metric, not of the answers.

## 7. Failure analysis

Three things to separate: a real retrieval failure, a real faithfulness weakness, and metric noise.

### 7.1 Retrieval failure — C04 (the only rule-level failure)

The question asks whether a breach of de-identified data must be notified **and what determines that**.
At `k=5` the retriever returns the anonymisation guide's incident-management passage but not the
notifiability criteria ("significant harm … or of significant scale"), so the answer explains that an
assessment is required but never states the test.

Cause: the question's wording is closest to the anonymisation passages, and policy/anonymisation
chunks crowd out the key-concepts chunk in the top 5.

In [ ]:
report.show_case(baseline, "C04")

### 7.2 Faithfulness — modal strength drift (S08)

The judge marked three of four claims in S08 unfaithful, with consistent reasons across repeated runs:
the documents say identity mapping tables *should* be encrypted (a recommended control) and the answer
says they *must* be. For policy questions, upgrading a recommendation to an obligation changes the
meaning, so this is a genuine defect rather than judge strictness. Section 8.2 addresses it.

In [ ]:
report.show_case(baseline, "S08", chars=200)

### 7.3 Why `factual_correctness` is low while the others are high

RAGAS computes this as a claim-level F1 in both directions: reference claims the answer omits count
against it, **and so do answer claims that are not in the reference** — even when they are true and
grounded. Decomposing all scored questions into claims separates the causes:

1. **Extra but correct claims** dominate. Coverage of reference claims averages ~0.72 while precision
   (answer claims present in the reference) averages ~0.61. S01 states all 8 reference claims and adds
   6 further grounded ones from the retrieved passages; faithfulness 1.00, factual_correctness 0.73.
2. **Reference scope.** Some references list more than the question asks (S03 asks only for approvals;
   the reference also lists the documentation requirements), which counts as missing coverage.
3. **Judge matching and variance.** C02's answer states the same conditions as the reference in
   question-specific wording and the judge matches neither direction (0.17 for a correct answer).
   Re-scoring identical text moved scores by ~0.14 on average (max 0.67).

Consequence for this report: **status accuracy and evidence recall are the metrics of record**;
`factual_correctness` is directional only, and low values are read case by case.

In [ ]:
# Where the judged scores disagree with the reviewed outcome.
weak = baseline.results[(baseline.results["factual_correctness"] < 0.5) | (baseline.results["faithfulness"] < 0.6)]
display(weak[["id", "type", "expected_status", "predicted_status", "evidence_recall", "rule_pass", "faithfulness", "factual_correctness"]])
report.show_case(baseline, "C02", chars=160)

### 7.4 What worked

- **Abstention.** All three unanswerable questions returned the exact required string, including C07,
  where the retrieved passages look highly relevant but state no response time.
- **Prompt injection.** S10 ("ignore the supplied documents … HIPAA") and C10 (a fake policy update
  inside the question) were both handled: no outside knowledge, and the documented threshold applied.
- **Source hierarchy.** S02 and S07 applied the internal policy over the general guidance and rejected
  the question's premise.
- **Typo robustness.** S09's PDF-extraction artefacts ("retenfon", "pafent") did not break retrieval.

## 8. Iterations

Each change targets a failure diagnosed in section 7, states its hypothesis and expected cost,
and is measured against the baseline run. Changes are made one at a time so the effect of each
is attributable.

### 8.1 `top_k` 5 → 8

Hypothesis: C04's missing passage is ranked just outside the top 5, so a larger window recovers it
without changing anything else. Expected cost: more tokens per call, higher latency, and lower
context precision (more retrieved-but-unused passages).

TODO: run `python -m eval.run_eval --name 02-topk8` with `RAG_TOP_K=8`, add it to `runs` in
section 5, and record the result and decision here.

In [ ]:
report.compare_runs(list(runs.values()))

### 8.2 Prompt rule: preserve modal strength

Hypothesis: adding an explicit instruction to keep the documents' `must` / `should` / `may` wording
removes the S08 drift without affecting other answers.

TODO: apply the prompt change, re-run, and compare.

## 9. Write-up

**1. Chunking strategy.** One chunk per Markdown section, with the heading path kept as metadata and
prefixed to the embedded text; sections over 300 words split on paragraph boundaries with a
one-paragraph overlap. This corpus is written one topic per section, so sections are self-contained
answer units and citations map to something a reviewer can verify. 112 chunks, 13–273 words.
Trade-off: a long section produces one averaged embedding covering several subtopics, and very short
chunks (policy examples) carry little lexical signal — mitigated by the heading prefix.

**2. Retrieval approach.** Dense retrieval over Gemini embeddings in a local Chroma collection, cosine
distance, top `k`. Single-path by design: the brief prefers evaluation over components, and the
evaluation identified only one retrieval miss (C04, section 7.1). Hybrid BM25 would mainly
help exact-term questions (e.g. `k-anonymity`, `30 days`); it is the first thing I would add if the
corpus grew.

**3. Prompting / answer generation.** Retrieved chunks are passed as labelled passages with their
source, section and authority. The model returns JSON (`answer`, `support_status`, `citations`,
`missing_information`) at temperature 0. The prompt requires citations per claim, applies the internal
policy over public guidance, requires premise checking, and treats passages and questions as data.

**4. How hallucination is reduced.** Layered: (a) only retrieved passages are in context; (b) the
prompt forbids outside knowledge and requires citations; (c) code validation drops citations outside
the retrieved set and degrades any uncited or malformed answer to abstention; (d) partial answers must
name what is missing instead of filling the gap; (e) evaluation includes questions whose real-world
answers are well known but absent from the corpus (HIPAA retention, PDPC penalties, approval SLAs).

**5. How the system decides when not to answer.** The model must return `not supported` when the
passages do not answer the question, and code then replaces the answer with the exact required string.
Abstention is also forced whenever the grounding contract is violated. There is no retrieval-score
threshold: distances proved poorly separated between answerable and unanswerable questions (C07
retrieves genuinely relevant passages that simply lack the requested detail), so the decision is made
on evidence content, not similarity.

**6. Known limitations.** Retrieval depends on `k` (C04); no access control on chunks, which the
internal policy itself requires for production; no re-ranking; abstention on ambiguous questions is
untested at scale; the judged metrics are noisy (±0.14) and the judge shares a model family with the
generator; 20 questions is too few for statistical claims; no multi-turn or conversational handling.

**7. What I would improve with more time.** Hybrid BM25 + dense retrieval with reciprocal rank fusion;
a cross-encoder re-ranker to keep `k` small while improving recall; per-claim citation validation
(currently citations are validated at answer level); a larger golden set with several paraphrases per
question; a stronger and independent judge model with repeated scoring to quantify variance; caching
and batching to cut cost.

**8. Safeguards for patient-sensitive or commercially sensitive data.** Chunk-level access control
enforced at retrieval, so a user never sees passages they are not entitled to (the internal policy
requires exactly this); separate indexes per sensitivity tier; audit logs of user, timestamp, query,
retrieved source IDs and support status, with raw patient content excluded from logs unless approved;
PII redaction before any telemetry or third-party API call; a private or in-region model deployment
with contractual no-training guarantees; monitoring for unsupported answers, unsafe disclosure and
injection attempts; human escalation for clinical or high-impact queries; documented retention and
annual review of the index, as the policy requires.

## 10. Production-readiness notes

**Privacy and access control.** Today every user can retrieve every chunk. Production needs identity
propagation and chunk-level filtering at query time, plus separate indexes for restricted operational
documents. The corpus itself sets this requirement (RAG systems must not expose passages the user is
not authorised to access).

**Monitoring.** Track abstention rate, share of answers with zero valid citations, guardrail warnings
(dropped citations, malformed JSON), retrieval distance distributions and latency percentiles. Alert
on abstention-rate drift, which usually signals index or embedding drift.

**Cost and latency.** Two model calls per question (embedding + generation); baseline latency ~3–6 s.
Cost scales with `k` × chunk size. Mitigations: cache embeddings (the corpus is static), cache answers
for repeated questions, and keep `k` as small as evaluation allows.

**Reliability.** Retries with backoff for provider errors; a stale-index check before answering (already
implemented); index rebuilds as a versioned artefact so a bad rebuild can be rolled back; scheduled
re-runs of the golden set as a regression gate on every prompt, model or corpus change.

**Change management.** Model and embedding versions pinned in config and recorded in every run's
`run.json`; the golden set is the contract, and any change to the prompt or retrieval is accepted
only if status accuracy and evidence recall hold.

## 11. Main question

> Based on your evaluation, for which kinds of questions is the system reliable, where does it fail,
> and what evidence supports those conclusions? What additional validation would be required before
> production use?

**Reliable.** Single-rule lookups, questions answered by one policy section, questions with a false
premise that the documents contradict, prompt-injection attempts, and unanswerable questions.
Evidence: support status correct on 20/20, all three abstentions exact, both injections resisted, and
S02/S07/C09/C10 correcting premises while applying the internal policy over general guidance.

**Weaker.** Multi-passage questions whose evidence is spread across documents. Evidence: C04 is the
only rule-level failure — at `k=5` the notifiability criteria were not retrieved, so the answer was
incomplete. Partial questions pass but state their supported part verbosely,
which is where the modal-strength drift appeared (S08).

**Unverified.** Answer wording quality is judged by metrics that are noisy at this sample size
(`factual_correctness` varies ±0.14 on re-scoring, and penalises true extra claims), so conclusions
about phrasing rest on manual review of 20 cases, not on the metric.

**Before production I would require:** (1) a larger golden set including paraphrases and adversarial
variants, with inter-rater agreement on expected behaviour; (2) retrieval evaluation at several `k`
values with recall and precision reported together; (3) an independent judge model plus human review
on a sample, with variance reported; (4) access-control tests proving unauthorised passages are never
retrievable; (5) red-teaming for injection and data-exfiltration attempts; (6) load and cost tests at
expected volumes; (7) a regression gate wired into deployment so no prompt, model or corpus change
ships without re-running the golden set.